# Trening NER na Colabie (T4) — mixed dataset, 3 modele

Fine-tuning na **zmiksowanym** datasecie: injection (formalne nazwy z baz) + golden-style
(Haiku, krótkie kliniczne formy) → `output/ner_dataset_mixed.jsonl`. 9 typów encji (EN):
PERSON, DISEASE, DRUG, TEST, HOSPITAL, ADDRESS, DATE, PESEL, PHONE → 19 klas IOB2.

| Model | Rozmiar | Oś porównania |
|---|---|---|
| `allegro/herbert-base-cased` | 124M | baseline PL (BERT) |
| `sdadas/polish-roberta-base-v2` | 124M | BERT vs RoBERTa (model w Space) |
| `xlm-roberta-base` | 278M | polski vs multilingual |

**Przed startem:** Runtime → Change runtime type → **T4 GPU**.

Każdy model ewaluowany na trzech zbiorach:
- **split** — wewnętrzny test mixed (ta sama dystrybucja co train; sanity, zawyża)
- **golden** — `test/dataset_1.json` (niezależny, human-style markup; before/after headline)
- **held-out** — `ner_dataset_golden_heldout.jsonl` (golden-style poza treningiem; generalizacja w stylu)

Wyniki + artefakty modeli w W&B (projekt `nlp-ner`).

In [ ]:
!nvidia-smi

In [ ]:
# idempotentne: re-run komórki nie zagnieżdża klonów (ścieżki absolutne)
import os
if not os.path.exists("/content/nlp-ner"):
    !git clone https://github.com/marek-olejniczak/nlp-ner.git /content/nlp-ner
%cd /content/nlp-ner
!git pull

In [ ]:
# torch jest preinstalowany na Colabie
!pip install -q transformers datasets seqeval accelerate wandb tqdm

In [ ]:
import wandb
wandb.login()  # wklej API key z https://wandb.ai/authorize

In [ ]:
# (model, batch, grad_accum) — wszystkie base'y mieszczą batch 16 na 16GB T4
MODELS = [
    ("allegro/herbert-base-cased", 16, 1),
    ("sdadas/polish-roberta-base-v2", 16, 1),
    ("xlm-roberta-base", 16, 1),
]

## Trening + ewaluacja w pętli

Każdy model: fine-tuning (3 epoki, lr 2e-5, fp16, najlepszy checkpoint wg F1
na walidacji) → ewaluacja na test splicie i na golden secie.
Paski tqdm pokazują postęp na bieżąco.

Najlepszy checkpoint każdego modelu jest automatycznie logowany jako **artefakt W&B**
(`<model>-ner`, type=model) — sesja Colaba jest ulotna, więc model nie zginie nawet
jak runtime padnie. Podgląd: zakładka *Artifacts* w projekcie `nlp-ner` na wandb.ai.

In [ ]:
DATA    = "output/ner_dataset_mixed.jsonl"
GOLDEN  = "test/dataset_1.json"                       # niezależny golden (markup)
HELDOUT = "output/ner_dataset_golden_heldout.jsonl"   # golden-style poza treningiem

for model, bs, accum in MODELS:
    short = model.split("/")[-1]
    print(f"\n{'='*70}\n  {model}\n{'='*70}")
    !python -m training.train --model {model} --data {DATA} --batch-size {bs} --grad-accum {accum} --fp16
    print("--- split (mixed, sanity) ---")
    !python -m training.evaluate --checkpoint models/{short}/best --data {DATA}
    print("--- golden (niezależny, human-style) ---")
    !python -m training.eval_set --input {GOLDEN} --checkpoint models/{short}/best
    print("--- held-out (golden-style, poza treningiem) ---")
    !python -m training.evaluate --checkpoint models/{short}/best --test-file {HELDOUT}

## Tabela zbiorcza

Micro F1 = wszystkie encje do jednego worka (dominują częste klasy).
Macro F1 = średnia po typach encji (wrażliwa na słabe klasy, np. DRUG).
Kolumna **golden** to F1 na niezależnym secie — spodziewaj się, że będzie niższy
od test split (split pochodzi z tej samej fabryki szablonów co train).

In [ ]:
import json
from pathlib import Path

def micro_f1(path):
    if not Path(path).exists():
        return None
    return json.loads(Path(path).read_text())["micro avg"]["f1-score"]

print(f"{'model':32s} {'split':>8s} {'golden':>8s} {'held-out':>9s}")
print("-" * 60)
for model, _, _ in MODELS:
    short = model.split("/")[-1]
    split   = micro_f1(f"models/{short}/best/eval_report_test.json")
    golden  = micro_f1(f"models/{short}/best/eval_report_golden.json")
    heldout = micro_f1(f"models/{short}/best/eval_report_ner_dataset_golden_heldout.json")
    fmt = lambda x: f"{x:.4f}" if x is not None else "   —  "
    print(f"{short:32s} {fmt(split):>8s} {fmt(golden):>8s} {fmt(heldout):>9s}")

## Wypchnięcie najlepszego modelu na HF Hub → Space sam go podbije

Wybierz zwycięzcę wg tabeli i wypchnij na `michaelo-ponteski/ner-medical-pl` — apka
[medical-text-anonymizer](https://huggingface.co/spaces/michaelo-ponteski/medical-text-anonymizer)
ładuje ten model z Huba (po pushu zrestartuj Space, by podebrał nową wersję).
Token z https://huggingface.co/settings/tokens (Write).

In [ ]:
from huggingface_hub import login
login()  # token Write z https://huggingface.co/settings/tokens

BEST = "polish-roberta-base-v2"   # podmień na zwycięzcę z tabeli
!python -m tools.push_to_hub --checkpoint models/{BEST}/best --repo-id michaelo-ponteski/ner-medical-pl

# Alternatywnie — pobierz checkpoint lokalnie (zip):
# !zip -rq {BEST}-ner.zip models/{BEST}/best
# from google.colab import files; files.download(f"{BEST}-ner.zip")